# `ai_parse_document` — Multi-Format Extraction & Accuracy Analysis

## Overview
This notebook benchmarks Databricks `ai_parse_document` (v2.0) against **4 document types**:

| File | Format | Key challenge |
|---|---|---|
| `sample_dashboard.png` | Image / PNG | Figure + text extraction from a raster image |
| `sample_invoice.pdf` | PDF (structured) | Tables, key-value pairs, numeric accuracy |
| `sample_presentation.pptx` | PowerPoint | Multi-slide layout, mixed text/image |
| `old_articles.pdf` | Scanned PDF | Handwritten / low-resolution OCR |

## Accuracy Metrics
For each file we measure:
- **Element coverage** — types of elements detected (text, table, figure, …)
- **Error rate** — `error_status` from the function output
- **Completeness score** — ratio of non-empty elements vs total pages
- **Confidence proxy** — average element confidence when available
- **Format-specific checks** — e.g. table row count for invoice, slide count for PPTX

In [0]:
%pip install -q pymupdf Pillow numpy landingai-ade --quiet
# PyMuPDF (fitz) replaces pdf2image: pure Python, no poppler binary required
dbutils.library.restartPython()


## 1. Configuration

In [0]:
%run ../_config/config_unity_catalog

In [0]:
import json
import time
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

# ── Paths ──────────────────────────────────────────────────────────────────
PATH_VOLUME   = f"/Volumes/{catalog}/{schema}/raw_data/multiformat"
PATH_IMAGES   = f"/Volumes/{catalog}/{schema}/raw_data/parsed_images"
PATH_RESULTS  = f"/Volumes/{catalog}/{schema}/raw_data/parse_results"

# Create output directories if they don't exist
dbutils.fs.mkdirs(PATH_IMAGES)
dbutils.fs.mkdirs(PATH_RESULTS)

# ── Files to analyse ───────────────────────────────────────────────────────
FILES = {
    "dashboard_png" : "sample_dashboard.png",
    "Invoice_jpg"   : "Invoice.jpg",
    "AccidentStatement_pdf" : "AccidentStatement.pdf",
    "old_articles"  : "old_articles.pdf",
}

print(f"Volume path   : {PATH_VOLUME}")
print(f"Images output : {PATH_IMAGES}")
print(f"Results output: {PATH_RESULTS}")
print(f"Files to parse: {list(FILES.values())}")

## 2. Parse All Documents with `ai_parse_document` v2.0

We use:
- `version = '2.0'` — latest schema (Jan 2026), based on `elements` array (replaces the legacy `pages` array)
- `descriptionElementTypes = '*'` — in v2.0, `'*'` and `'figure'` produce **the same behaviour**: AI descriptions are generated for **figures only** (descriptions for tables and text blocks are not yet supported)
- `imageOutputPath` — persist rendered page images to the volume for visual inspection or multi-modal RAG

In [0]:
parse_results = {}
parse_timings = {}

for label, filename in FILES.items():
    file_path = f"{PATH_VOLUME}/{filename}"
    print(f"\nParsing [{label}] → {filename}")
    
    t0 = time.time()
    sql = f"""
        WITH parsed AS (
            SELECT
                path,
                ai_parse_document(
                    content,
                    map(
                        'version',                '2.0',
                        'imageOutputPath',        '{PATH_IMAGES}/{label}/',
                        'descriptionElementTypes','*'
                    )
                ) AS parsed
            FROM READ_FILES('{file_path}', format => 'binaryFile')
        )
        SELECT
            path,
            to_json(parsed)                         AS parsed_json,
            parsed:error_status                     AS error_status,
            parsed:metadata:version                   AS version,
            size(cast(parsed:document:elements as array<variant>)) AS element_count
        FROM parsed
    """
    
    df = spark.sql(sql)
    row = df.collect()[0]
    elapsed = round(time.time() - t0, 2)
    parse_timings[label] = elapsed
    
    parsed_data = json.loads(row["parsed_json"])
    parse_results[label] = {
        "filename"      : filename,
        "path"          : row["path"],
        "error_status"  : None if (row["error_status"] is None or str(row["error_status"]).strip('"') in ("null","")) else str(row["error_status"]),
        "version"       : row["version"],
        "element_count" : row["element_count"],
        "parsed"        : parsed_data,
        "elapsed_s"     : elapsed,
    }
    
    status = "" if row["error_status"] is None else f"  {row['error_status']}"
    print(f"   {status}  |  version : {row['version']}  |  elements: {row['element_count']}  |  {elapsed}s")

print("\nAll documents parsed.")

## 2b. Markdown Reconstruction

Reconstruct each parsed document as **readable Markdown** from its elements array.

The `elements_to_markdown()` function:
- Sorts elements by `page_id` → `bbox.y_top` to respect reading order
- Maps each `type` to its Markdown equivalent (`title` → `#`, `section_header` → `##`, etc.)
- Converts inline HTML `<table>` content to a proper Markdown table
- Renders `figure` elements as a blockquote with the AI-generated description when available
- Inserts `---` page breaks between pages
- Saves each reconstruction as a `.md` file in the results volume

In [0]:
import re
from html.parser import HTMLParser

# ── HTML table → Markdown table ────────────────────────────────────────────
class _TableParser(HTMLParser):
    """Minimal HTML table parser → list of rows (each row = list of cell strings)."""
    def __init__(self):
        super().__init__()
        self.rows, self._row, self._cell, self._in_cell = [], [], [], False
    def handle_starttag(self, tag, attrs):
        if tag in ('tr',):  self._row = []
        if tag in ('td','th'): self._in_cell, self._cell = True, []
    def handle_endtag(self, tag):
        if tag in ('td','th') and self._in_cell:
            self._row.append(' '.join(self._cell).strip())
            self._in_cell = False
        if tag == 'tr' and self._row:
            self.rows.append(self._row)
    def handle_data(self, data):
        if self._in_cell: self._cell.append(data.strip())

def html_table_to_md(html: str) -> str:
    p = _TableParser()
    p.feed(html)
    if not p.rows: return html
    header  = p.rows[0]
    sep     = ['---'] * len(header)
    body    = p.rows[1:]
    lines   = ['| ' + ' | '.join(header) + ' |',
               '| ' + ' | '.join(sep)    + ' |']
    for row in body:
        # Pad short rows
        padded = row + [''] * (len(header) - len(row))
        lines.append('| ' + ' | '.join(padded[:len(header)]) + ' |')
    return '\n'.join(lines)

# ── Type → Markdown prefix map ──────────────────────────────────────────────
TYPE_PREFIX = {
    'title'          : '# ',
    'section_header' : '## ',
    'page_header'    : '> **',   # blockquote for headers/footers
    'page_footer'    : '> *',
    'caption'        : '_',
    'text'           : '',
    'figure'         : None,     # handled separately
    'table'          : None,     # handled separately
}

# ── Core reconstruction function ───────────────────────────────────────────
def elements_to_markdown(elements: list, doc_title: str = '') -> str:
    """Convert a v2.0 elements list to Markdown, preserving reading order."""
    if not elements:
        return f'# {doc_title}\n\n> ⚠️ No elements extracted by ai_parse_document.\n'

    # Sort: page first, then top-y of first bbox
    def sort_key(e):
        bbox = e.get('bbox', [])
        page = bbox[0]['page_id'] if bbox and 'page_id' in bbox[0] else 0
        y    = bbox[0]['coord'][1] if bbox and len(bbox[0].get('coord', [])) >= 2 else 0
        return (page, y)

    sorted_elems = sorted(elements, key=sort_key)

    lines        = [f'# {doc_title}', ''] if doc_title else []
    current_page = -1

    for elem in sorted_elems:
        bbox       = elem.get('bbox', [])
        page       = bbox[0]['page_id'] if bbox and 'page_id' in bbox[0] else 0
        etype      = elem.get('type', 'text')
        content    = (elem.get('content') or '').strip()
        description = (elem.get('description') or '').strip()

        # Page break
        if page != current_page:
            if current_page >= 0:
                lines += ['', '---', f'', f'*— Page {page + 1} —*', '']
            current_page = page

        # Render by type
        if etype == 'table':
            if content.startswith('<table'):
                lines.append(html_table_to_md(content))
            else:
                lines.append(content)

        elif etype == 'figure':
            if description:
                lines.append(f'> 📊 *{description}*')
            if content:  # OCR text found inside the figure bbox
                lines.append(f'```\n{content}\n```')

        elif etype in ('page_header', 'page_footer'):
            prefix  = TYPE_PREFIX[etype]
            suffix  = '**' if etype == 'page_header' else '*'
            lines.append(f'{prefix}{content}{suffix}')

        elif etype == 'caption':
            lines.append(f'_{content}_')

        elif etype == 'title':
            lines.append(f'# {content}')

        elif etype == 'section_header':
            lines.append(f'## {content}')

        else:  # text and anything unknown
            lines.append(content)

        lines.append('')  # blank line between every element

    return '\n'.join(lines)


# ── Generate & display Markdown for all documents ──────────────────────────
markdowns = {}

for label, result in parse_results.items():
    elements = result['parsed'].get('document', {}).get('elements', [])
    md = elements_to_markdown(elements, doc_title=result['filename'])
    markdowns[label] = md

    # Save to volume
    out_path = f"{PATH_RESULTS}/{label}.md"
    with open(out_path, 'w', encoding='utf-8') as f:
        f.write(md)

    line_count = md.count('\n')
    char_count = len(md)
    print(f"[{label}] → {out_path}")
    print(f"   {len(elements):>3} elements  |  {line_count:>4} lines  |  {char_count:>6} chars")
    print()

print("All Markdown reconstructions saved.")


In [0]:
# ── Preview in notebook (pick one label to display) ────────────────────────
PREVIEW_LABEL = "AccidentStatement_pdf"    # change to: dashboard_png | presentation | old_articles

md_preview = markdowns[PREVIEW_LABEL]

# Render as HTML inside the notebook cell output
try:
    import markdown as md_lib
except ImportError:
    # markdown not available — fallback to plain <pre> display
    md_lib = None

if md_lib:
    html_out = md_lib.markdown(md_preview, extensions=['tables'])
else:
    # Simple pre-formatted fallback
    escaped  = md_preview.replace('&','&amp;').replace('<','&lt;').replace('>','&gt;')
    html_out = f'<pre style="font-family:monospace;font-size:13px;line-height:1.5">{escaped}</pre>'

displayHTML(f'''
<div style="
  max-width:900px; margin:auto; padding:32px;
  font-family: Georgia, serif; font-size:15px; line-height:1.7;
  background:#fafafa; border:1px solid #ddd; border-radius:8px;
">
  <p style="font-size:11px;color:#888;margin-bottom:24px;">
    Markdown reconstruction — <strong>{PREVIEW_LABEL}</strong>
  </p>
  {html_out}
</div>
''')


## 4. Element-Level Analysis

Extract the `elements` array from each document and build a flat DataFrame to analyse element types, confidence, and content.

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, BooleanType, DoubleType

# ── Explicit schema: avoids CANNOT_DETERMINE_TYPE when all values are None ─
# Note: ai_parse_document v2.0 uses 'content' (not 'text') and 'page_id' is
# nested inside bbox[0]['page_id'], not a top-level field.
SCHEMA = StructType([
    StructField("file_label",      StringType(),  False),
    StructField("filename",        StringType(),  False),
    StructField("element_type",    StringType(),  False),
    StructField("page_number",     IntegerType(), True),
    StructField("has_content",     BooleanType(), False),
    StructField("has_description", BooleanType(), False),
    StructField("has_confidence",  BooleanType(), False),
    StructField("confidence",      DoubleType(),  True),   # always None in v2.0 — typed explicitly
    StructField("content_length",  IntegerType(), False),
])

# ── Flatten elements from all files ────────────────────────────────────────
all_elements = []

for label, result in parse_results.items():
    doc      = result["parsed"].get("document", {})
    elements = doc.get("elements", [])

    for elem in elements:
        # page_id is nested inside bbox list in v2.0
        bbox        = elem.get("bbox", [])
        page_number = int(bbox[0]["page_id"]) if bbox and "page_id" in bbox[0] else None

        # v2.0 uses 'content', not 'text'
        content     = elem.get("content") or ""
        description = elem.get("description") or ""
        conf_raw    = elem.get("confidence")          # None in current v2.0 model

        all_elements.append((
            label,
            result["filename"],
            str(elem.get("type", "unknown")),
            page_number,
            bool(content.strip()),
            bool(description.strip()),
            conf_raw is not None,
            float(conf_raw) if conf_raw is not None else None,
            len(content),
        ))

df_elements = spark.createDataFrame(all_elements, schema=SCHEMA)
df_elements.createOrReplaceTempView("elements")

print(f"Total elements across all files: {df_elements.count()}")
display(df_elements.groupBy("file_label", "element_type").count().orderBy("file_label", "count", ascending=[True, False]))

## 5. Per-File Accuracy Scorecard

We compute a **composite accuracy score** based on:

| Sub-score | Weight | Description |
|---|---|---|
| `no_error`         | 30% | `error_status` is null |
| `element_coverage` | 30% | At least 1 element extracted per page |
| `text_fill_rate`   | 20% | % of elements containing non-empty text |
| `avg_confidence`   | 20% | Average confidence score (when available) |

In [0]:
scorecards = []

for label, result in parse_results.items():
    elements = result["parsed"].get("document", {}).get("elements", [])
    page_count = max(int(str(result.get("page_count") or 1).strip('"')), 1)
    elem_count = len(elements)
    
    # Sub-scores ─────────────────────────────────────────────────────────────
    # 1. No error
    no_error = 1.0 if result["error_status"] is None else 0.0

    # 2. Element coverage: at least 1 element per page (capped at 1.0)
    element_coverage = min(elem_count / max(int(page_count), 1), 1.0)

    # 3. Text fill rate: % of elements with non-empty text
    if elem_count > 0:
        text_elems = sum(1 for e in elements if (e.get("content") or "").strip())
        text_fill_rate = text_elems / elem_count
    else:
        text_fill_rate = 0.0

    # 4. Average confidence (default 0.75 when not provided by model)
    confidences = [float(e["confidence"]) for e in elements if "confidence" in e]
    avg_confidence = sum(confidences) / len(confidences) if confidences else 0.75

    # Composite score
    score = (
        0.30 * no_error +
        0.30 * element_coverage +
        0.20 * text_fill_rate +
        0.20 * avg_confidence
    ) * 100

    # Element type breakdown
    type_counts = {}
    for e in elements:
        t = e.get("type", "unknown")
        type_counts[t] = type_counts.get(t, 0) + 1

    scorecards.append({
        "file_label"       : label,
        "filename"         : result["filename"],
        "error_status"     : result["error_status"] or "none",
        "page_count"       : int(page_count),
        "element_count"    : elem_count,
        "no_error"         : round(no_error * 100, 1),
        "element_coverage" : round(element_coverage * 100, 1),
        "text_fill_rate"   : round(text_fill_rate * 100, 1),
        "avg_confidence"   : round(avg_confidence * 100, 1),
        "composite_score"  : round(score, 1),
        "elapsed_s"        : result["elapsed_s"],
        "element_types"    : json.dumps(type_counts),
    })

from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType
SCORE_SCHEMA = StructType([
    StructField("file_label",       StringType(),  False),
    StructField("filename",         StringType(),  False),
    StructField("error_status",     StringType(),  True),
    StructField("page_count",       IntegerType(), False),
    StructField("element_count",    IntegerType(), False),
    StructField("no_error",         FloatType(),   False),
    StructField("element_coverage", FloatType(),   False),
    StructField("text_fill_rate",   FloatType(),   False),
    StructField("avg_confidence",   FloatType(),   False),
    StructField("composite_score",  FloatType(),   False),
    StructField("elapsed_s",        FloatType(),   False),
    StructField("element_types",    StringType(),  True),
])
score_rows = [tuple(s.values()) for s in scorecards]
df_scorecard = spark.createDataFrame(score_rows, schema=SCORE_SCHEMA)
df_scorecard.createOrReplaceTempView("scorecard")

display(
    df_scorecard.select(
        "filename", "page_count", "element_count",
        "no_error", "element_coverage", "text_fill_rate", "avg_confidence",
        "composite_score", "elapsed_s"
    ).orderBy(F.col("composite_score").desc())
)

## 6. Format-Specific Deep Dives

### 5a. AccidentStatement PDF — Table & Key-Value Extraction

In [0]:
invoice_elements = parse_results["AccidentStatement_pdf"]["parsed"].get("document", {}).get("elements", [])
tables  = [e for e in invoice_elements if e.get("type") == "table"]
# Key-value pairs: text elements whose content contains ':'
kv_text = [e for e in invoice_elements if e.get("type") == "text" and ":" in (e.get("content") or "")]

print(f"Invoice analysis")
print(f"   Tables detected        : {len(tables)}")
print(f"   Key-value text blocks  : {len(kv_text)}")

if tables:
    for i, t in enumerate(tables):
        desc    = (t.get("description") or "(no description)")[:100]
        content = t.get("content") or ""
        # Count <tr> occurrences as a proxy for row count in HTML table content
        row_count = content.count("<tr>") - 1  # subtract header row
        print(f"   Table {i+1}: ~{row_count} data rows | {desc}")

print("\nSample key-value extractions:")
for kv in kv_text[:5]:
    print(f"   {(kv.get('content') or '').strip()[:120]}")

### 5b. AccidentStatement — LandingAI ADE vs `ai_parse_document` Head-to-Head

The `AccidentStatement.pdf` is a European insurance form (*constat amiable*) — one of the hardest document types for parsing tools:

| Challenge | Detail |
|---|---|
| **Two-column layout** | Vehicle A (left) and Vehicle B (right) with shared centre |
| **17 checkboxes** | Numbered circumstances list — requires form understanding |
| **Handwritten fields** | Names, addresses, dates, remarks filled by hand |
| **Accident sketch** | Hand-drawn diagram with vehicle positions |
| **Mixed content** | Printed labels + handwritten values in same cell |

We run both tools on the **original PDF** (no preprocessing needed — it is a clean scan) and compare extraction quality side by side.

In [0]:
import os, time, json
from pathlib import Path
from landingai_ade import LandingAIADE

ACCIDENT_PDF = f"{PATH_VOLUME}/AccidentStatement.pdf"

# ── 1. ai_parse_document result (already in parse_results) ────────────────
dbx_elems  = parse_results['AccidentStatement_pdf']['parsed'].get('document',{}).get('elements',[])
dbx_err    = parse_results['AccidentStatement_pdf']['error_status']
dbx_time   = parse_results['AccidentStatement_pdf']['elapsed_s']

print(f"Databricks ai_parse_document:")
print(f"  Elements : {len(dbx_elems)}")
print(f"  Error    : {dbx_err}")
print(f"  Time     : {dbx_time}s")


# ── 2. LandingAI ADE ──────────────────────────────────────────────────────
try:
    LANDING_AI_KEY = dbutils.secrets.get(scope='landing_ai', key='vision_agent_api_key')
except Exception:
    LANDING_AI_KEY = os.environ.get('vision_agent_api_key', 'YOUR_API_KEY_HERE')

client = LandingAIADE(apikey=LANDING_AI_KEY)
GROUNDING_ACCIDENT = f"{PATH_RESULTS}/landingai_accident_groundings"
dbutils.fs.mkdirs(GROUNDING_ACCIDENT)

print(f"\nParsing AccidentStatement with LandingAI ADE (dpt-2-latest) ...")
t0 = time.time()
ade_resp     = client.parse(document=Path(ACCIDENT_PDF), model='dpt-2-latest',
                             save_to=GROUNDING_ACCIDENT)
ade_time     = round(time.time() - t0, 2)
ade_chunks   = ade_resp.chunks
ade_markdown = ade_resp.markdown or ''

print(f"  Chunks   : {len(ade_chunks)}")
print(f"  MD chars : {len(ade_markdown):,}")
print(f"  Time     : {ade_time}s")


In [0]:
for chunk in ade_chunks[:2]: print(chunk.markdown)

In [0]:
# ── 3. Scoring helper ─────────────────────────────────────────────────────
def score_dbx(elements, error_status, elapsed_s):
    n   = len(elements)
    no_err   = 1.0 if error_status is None else 0.0
    fill     = sum(1 for e in elements if (e.get('content') or '').strip()) / n if n else 0.0
    confs    = [float(e['confidence']) for e in elements if 'confidence' in e]
    avg_conf = sum(confs)/len(confs) if confs else 0.75
    composite = (0.33*no_err + 0.33*fill + 0.33*avg_conf) * 100
    types = {}
    for e in elements: t=e.get('type','?'); types[t]=types.get(t,0)+1
    return {'n':n,'no_err':round(no_err*100,1),
            'fill':round(fill*100,1),'conf':round(avg_conf*100,1),
            'score':round(composite,1),'elapsed':elapsed_s,'types':types}

def score_ade(chunks, elapsed_s):
    n       = len(chunks)
    filled  = sum(1 for c in chunks if (getattr(c, 'markdown', '') or '').strip())
    fill    = round(filled/n*100,1) if n else 0.0
    confs   = [c.confidence for c in chunks if hasattr(c,'confidence') and c.confidence is not None]
    avg_conf = round(sum(confs)/len(confs)*100,1) if confs else 75.0
    composite = round(0.33*100 + 0.33*fill + 0.33*avg_conf, 1)
    types = {}
    for c in chunks: t=str(c.type).lower(); types[t]=types.get(t,0)+1
    return {'n':n,'no_err':100.0,'fill':fill,'conf':avg_conf,
            'score':composite,'elapsed':elapsed_s,'types':types}

s_dbx = score_dbx(dbx_elems, dbx_err, dbx_time)
s_ade = score_ade(ade_chunks, ade_time)


# ── 4. Head-to-head table ─────────────────────────────────────────────────
print()
print('=' * 65)
print('  HEAD-TO-HEAD  --  AccidentStatement.pdf')
print('  (European insurance form: checkboxes, 2-col layout, handwriting)')
print('=' * 65)
hdr = '{:<26} {:>16} {:>16}'
print(hdr.format('Metric', 'Databricks v2.0', 'LandingAI ADE'))
print('-' * 65)
rows = [
    ('Elements / chunks',  s_dbx['n'],      s_ade['n'],      ''),
    ('No-error',           s_dbx['no_err'], s_ade['no_err'], '%'),
    ('Text fill rate',     s_dbx['fill'],   s_ade['fill'],   '%'),
    ('Avg confidence',     s_dbx['conf'],   s_ade['conf'],   '%'),
    ('Composite score',    s_dbx['score'],  s_ade['score'],  '%'),
    ('Parse time',         s_dbx['elapsed'],s_ade['elapsed'],'s'),
]
for name, dbx_v, ade_v, unit in rows:
    if name == 'Parse time':
        winner = 'DBX <<' if float(dbx_v) <= float(ade_v) else 'ADE <<'
    else:
        winner = 'DBX <<' if float(dbx_v) >= float(ade_v) else 'ADE <<'
    print(hdr.format(name, f'{dbx_v}{unit}', f'{ade_v}{unit}') + f'  {winner}')
print('=' * 65)

# Element type distribution
all_types = sorted(set(list(s_dbx['types'].keys()) + list(s_ade['types'].keys())))
print('\nElement / chunk type distribution:')
hdr2 = '{:<22} {:>16} {:>16}'
print(hdr2.format('Type', 'Databricks v2.0', 'LandingAI ADE'))
print('-' * 56)
for t in all_types:
    print(hdr2.format(t, s_dbx['types'].get(t,0), s_ade['types'].get(t,0)))

#### Side-by-Side Markdown Preview — AccidentStatement

In [0]:
# Build Databricks markdown for AccidentStatement
dbx_accident_md = elements_to_markdown(dbx_elems, 'AccidentStatement.pdf')

# Save both outputs
for path, content in [
    (f'{PATH_RESULTS}/accident_databricks.md', dbx_accident_md),
    (f'{PATH_RESULTS}/accident_landingai.md',  ade_markdown),
]:
    with open(path, 'w', encoding='utf-8') as f: f.write(content)
    print(f'Saved: {path}')

def esc(s): return s.replace('&','&amp;').replace('<','&lt;').replace('>','&gt;')

# Highlight form-specific extractions for quick visual check
# Look for key fields we know are in the document
EXPECTED_FIELDS = ['Doe', 'Jane', 'Smith', 'Jade', 'Renault', 'Fiat',
                   '09.10.2024', 'CDE Assurance', 'ZYX Assurance',
                   'rear right bumper', 'front right bumper',
                   'was on his phone', 'Limmatstrasse']

print('\nField extraction check (known ground-truth values):')
print(f"{'Field':<30} {'Databricks':>12} {'LandingAI':>12}")
print('-' * 56)
dbx_text = ' '.join((e.get('content') or '') for e in dbx_elems).lower()
ade_text = ade_markdown.lower()
for field in EXPECTED_FIELDS:
    in_dbx = 'FOUND' if field.lower() in dbx_text else 'MISSING'
    in_ade = 'FOUND' if field.lower() in ade_text else 'MISSING'
    flag   = '' if in_dbx == in_ade else '  <-- differs'
    print(f"{field:<30} {in_dbx:>12} {in_ade:>12}{flag}")

displayHTML(f'''
<div style="display:flex;gap:20px;font-family:Georgia,serif;font-size:13px;line-height:1.6;">
  <div style="flex:1;padding:18px;background:#f5f9ff;border:1px solid #b3d1ff;border-radius:8px;">
    <p style="font-weight:bold;color:#1a4a9e;margin:0 0 10px">
      Databricks ai_parse_document v2.0 &nbsp;
      <small style="font-weight:normal">{s_dbx['n']} elements &middot; {s_dbx['score']}%</small>
    </p>
    <pre style="font-size:12px;white-space:pre-wrap;max-height:550px;overflow-y:auto;margin:0">{esc(dbx_accident_md[:4000])}</pre>
  </div>
  <div style="flex:1;padding:18px;background:#f5fff8;border:1px solid #7ecfaa;border-radius:8px;">
    <p style="font-weight:bold;color:#1a6640;margin:0 0 10px">
      LandingAI ADE &mdash; dpt-2-latest &nbsp;
      <small style="font-weight:normal">{s_ade['n']} chunks &middot; {s_ade['score']}%</small>
    </p>
    <pre style="font-size:12px;white-space:pre-wrap;max-height:550px;overflow-y:auto;margin:0">{esc(ade_markdown[:4000])}</pre>
  </div>
</div>
''')


### 5e. Scanned PDF (old_articles) — OCR Quality Indicators

In [0]:
scan_elements = parse_results["old_articles"]["parsed"].get("document", {}).get("elements", [])

# ── Handle case where model returns 0 elements (e.g. very old/degraded scans) ──
if not scan_elements:
    print("Scanned PDF — OCR Quality Report")
    print("   WARNING: 0 elements extracted.")
    print("   This typically means the scan resolution is too low (<150 DPI),")
    print("   the document language/script is not supported, or the file is")
    print("   image-only with no detectable text regions.")
    print("   Recommendation: pre-process with PIL/OpenCV before re-parsing.")
else:

    import re
    garbled_pattern = re.compile(r'[^\x00-\x7F]{3,}|[\x00-\x08\x0b\x0e-\x1f]{2,}')

    total_text_chars = 0
    garbled_blocks   = 0
    empty_text       = 0
    samples          = []

    for e in scan_elements:
        txt = e.get("text", "")
        total_text_chars += len(txt)
        if not txt.strip():
            empty_text += 1
        elif garbled_pattern.search(txt):
            garbled_blocks += 1
        if txt.strip() and len(samples) < 3:
            samples.append(txt.strip()[:200])

    n = len(scan_elements) or 1
    ocr_health = round((1 - (garbled_blocks + empty_text) / n) * 100, 1)

    print("📰 Scanned PDF — OCR Quality Report")
    print(f"   Total elements     : {n}")
    print(f"   Empty text blocks  : {empty_text}  ({round(empty_text/n*100,1)}%)")
    print(f"   Garbled blocks     : {garbled_blocks}  ({round(garbled_blocks/n*100,1)}%)")
    print(f"   Total chars read   : {total_text_chars:,}")
    print(f"   OCR health score   : {ocr_health}% ({'🟢 Good' if ocr_health > 75 else '🟡 Medium' if ocr_health > 50 else '🔴 Poor'})")
    print("\n📝 Sample extracted text:")
    for i, s in enumerate(samples, 1):
        print(f"   [{i}] {s}")

## 6. Scanned Document Deep Dive -- `old_articles.pdf`

The baseline parse returned **0 elements** because `old_articles.pdf` is a 1908 French journal scanned at very low resolution. This section applies a full preprocessing pipeline to rescue the document before re-parsing.

### 6a. PIL Preprocessing Pipeline

Three steps applied per page:

| Step | Technique | Purpose |
|---|---|---|
| **Upscale** | PIL `LANCZOS` resize to 300 DPI equivalent | More pixels for OCR |
| **Binarise** | Otsu global threshold via numpy | Removes grey noise, sharp B&W |
| **Deskew** | Projection-profile angle scan | Corrects tilt from scanning |

The preprocessed pages are re-assembled into a new PDF and saved to the volume.

In [0]:
import io, time
import numpy as np
import fitz                              # PyMuPDF — no poppler required
from PIL import Image, ImageFilter

SOURCE_PDF       = f"{PATH_VOLUME}/old_articles.pdf"
PREPROCESSED_PDF = f"{PATH_VOLUME}/old_articles_preprocessed.pdf"
TARGET_DPI       = 300
ZOOM             = TARGET_DPI / 72.0     # fitz renders at 72 dpi by default

pdf_doc = fitz.open(SOURCE_PDF)
print(f"Source PDF : {SOURCE_PDF}")
print(f"Pages      : {pdf_doc.page_count}  |  target DPI: {TARGET_DPI}")


def deskew(pil_img, angle_range=5.0, steps=100):
    """Projection-profile deskew: maximise row-sum variance to find skew angle."""
    gray = np.array(pil_img.convert('L'))
    best_angle, best_score = 0.0, -1.0
    for angle in np.linspace(-angle_range, angle_range, steps):
        rotated  = Image.fromarray(gray).rotate(angle, expand=False, fillcolor=255)
        row_sums = np.array(rotated).sum(axis=1).astype(float)
        score    = float(np.var(row_sums))
        if score > best_score:
            best_score, best_angle = score, angle
    if abs(best_angle) > 0.1:
        return pil_img.rotate(best_angle, expand=False, fillcolor=255,
                               resample=Image.BICUBIC)
    return pil_img


def binarise(pil_img):
    """Otsu global threshold binarisation (pure numpy, no OpenCV)."""
    gray    = np.array(pil_img.convert('L'))
    hist, _ = np.histogram(gray.flatten(), 256, [0, 256])
    hist    = hist.astype(float)
    total   = gray.size
    sum_all = float((np.arange(256) * hist).sum())
    wb = sb  = 0.0
    best_thresh, best_var = 0, 0.0
    for t in range(256):
        wb += hist[t]; wf = total - wb
        if wb == 0 or wf == 0: continue
        sb += t * hist[t]
        mb, mf = sb / wb, (sum_all - sb) / wf
        var = wb * wf * (mb - mf) ** 2
        if var > best_var:
            best_var, best_thresh = var, t
    return Image.fromarray((gray > best_thresh).astype('uint8') * 255).convert('RGB')


# ── Rasterise + preprocess each page with fitz ─────────────────────────────
processed_pages = []
mat = fitz.Matrix(ZOOM, ZOOM)            # scale matrix for target DPI

for page_num in range(pdf_doc.page_count):
    t0   = time.time()
    page = pdf_doc.load_page(page_num)
    pix  = page.get_pixmap(matrix=mat, colorspace=fitz.csRGB)

    # fitz pixmap -> PIL Image
    raw_img = Image.frombytes('RGB', [pix.width, pix.height], pix.samples)
    w0, h0  = raw_img.size

    # Step 1: sharpen to recover edge detail lost in scan
    sharpened = raw_img.filter(ImageFilter.UnsharpMask(radius=1.5, percent=120, threshold=3))
    # Step 2: binarise (Otsu)
    binary    = binarise(sharpened)
    # Step 3: deskew
    deskewed  = deskew(binary)

    processed_pages.append(deskewed)
    print(f"  Page {page_num+1}: {w0}x{h0}px -> preprocessed in {round(time.time()-t0,2)}s")

pdf_doc.close()

# ── Re-assemble into PDF ────────────────────────────────────────────────────
processed_pages[0].save(
    PREPROCESSED_PDF, save_all=True,
    append_images=processed_pages[1:],
    format='PDF', resolution=TARGET_DPI,
)
import os
print(f"\nPreprocessed PDF saved : {PREPROCESSED_PDF}")
print(f"Size                   : {os.path.getsize(PREPROCESSED_PDF)//1024} KB")
print(f"Pages                  : {len(processed_pages)}")


### 6b. Re-parse Preprocessed PDF with `ai_parse_document`

Re-run `ai_parse_document v2.0` on the preprocessed PDF and compare the scorecard against the raw baseline.

In [0]:
import time, json

print('Parsing preprocessed PDF with ai_parse_document v2.0 ...')
t0 = time.time()

sql_preprocessed = f"""
    WITH parsed AS (
        SELECT path,
               ai_parse_document(content, map(
                   'version','2.0',
                   'imageOutputPath','{PATH_IMAGES}/old_articles_preprocessed/',
                   'descriptionElementTypes','*'
               )) AS parsed
        FROM READ_FILES('{PREPROCESSED_PDF}', format => 'binaryFile')
    )
    SELECT path, to_json(parsed) AS parsed_json,
           parsed:error_status AS error_status,
           parsed:metadata:page_count AS page_count,
           size(cast(parsed:document:elements as array<variant>)) AS element_count
    FROM parsed
"""

df_pre    = spark.sql(sql_preprocessed)
row_pre   = df_pre.collect()[0]
elapsed_pre = round(time.time() - t0, 2)
parsed_pre  = json.loads(row_pre['parsed_json'])
elems_pre   = parsed_pre.get('document', {}).get('elements', [])
err_pre     = None if (row_pre['error_status'] is None or str(row_pre['error_status']).strip('"') in ('null','')) else str(row_pre['error_status'])

print(f'  error_status : {err_pre}')
print(f"  page_count   : {row_pre['page_count']}")
print(f'  elements     : {len(elems_pre)}')
print(f'  elapsed      : {elapsed_pre}s')


def compute_score(elements, error_status, elapsed_s, label):
    n   = len(elements)
    no_error         = 1.0 if error_status is None else 0.0
    text_fill_rate   = sum(1 for e in elements if (e.get('content') or '').strip()) / n if n > 0 else 0.0
    confidences      = [float(e['confidence']) for e in elements if 'confidence' in e]
    avg_confidence   = sum(confidences)/len(confidences) if confidences else 0.75
    composite        = (0.30*no_error + 0.30*element_coverage + 0.20*text_fill_rate + 0.20*avg_confidence) * 100
    type_counts = {}
    for e in elements:
        t = e.get('type','unknown'); type_counts[t] = type_counts.get(t,0) + 1
    return {'label':label,'elements':n,
            'no_error':round(no_error*100,1),
            'text_fill_rate':round(text_fill_rate*100,1),'avg_confidence':round(avg_confidence*100,1),
            'composite_score':round(composite,1),'elapsed_s':elapsed_s,'type_counts':type_counts}

baseline_elems = parse_results['old_articles']['parsed'].get('document',{}).get('elements',[])
score_raw = compute_score(baseline_elems, parse_results['old_articles']['error_status'],
                          parse_results['old_articles']['elapsed_s'], 'Raw scan (baseline)')
score_pre = compute_score(elems_pre, err_pre, elapsed_pre, 'Preprocessed (PIL)')

print()
print('=' * 62)
print('  BEFORE / AFTER  --  ai_parse_document on old_articles.pdf')
print('=' * 62)
fmt = '{:<28} {:>7} {:>7} {:>8} {:>4}'
print(fmt.format('Metric','Before','After','Delta',''))
print('-' * 62)
metrics = [
    ('Elements extracted',  score_raw['elements'],          score_pre['elements'],          ''),
    ('No-error score',      score_raw['no_error'],          score_pre['no_error'],          '%'),
    ('Text fill rate',      score_raw['text_fill_rate'],    score_pre['text_fill_rate'],    '%'),
    ('Composite score',     score_raw['composite_score'],   score_pre['composite_score'],   '%'),
    ('Parse time',          score_raw['elapsed_s'],         score_pre['elapsed_s'],         's'),
]
for name, before, after, unit in metrics:
    delta = after - before; sign = '+' if delta > 0 else ''
    grade = 'UP' if delta > 0 else ('DN' if delta < 0 else '--')
    if name == 'Parse time': grade = 'OK' if delta <= 0 else 'SLW'
    print(fmt.format(name, f'{before}{unit}', f'{after}{unit}', f'{sign}{round(delta,1)}{unit}', grade))
print('=' * 62)
print()
print('Element type breakdown (preprocessed):')
for t, c in sorted(score_pre['type_counts'].items(), key=lambda x:-x[1]):
    print(f'  {t:<20} {c}')

preprocessed_parse_score = score_pre
preprocessed_parse_elems = elems_pre


### 6c. LandingAI ADE vs Databricks `ai_parse_document` -- Head-to-Head

Parse the **same preprocessed PDF** with LandingAI's Agentic Document Extraction (ADE) using the `landingai-ade` Python library and model `dpt-2-latest`.

**Setup required:**  
Store your key as a Databricks secret:  
`dbutils.secrets.put(scope='landing_ai', key='VISION_AGENT_API_KEY', string_value='...')`

| Dimension | Description |
|---|---|
| **Extraction yield** | Chunk count and type distribution |
| **Text fill rate** | % of chunks with non-empty text |
| **Markdown quality** | Character count and structural richness |
| **Latency** | Wall-clock parse time |
| **Composite score** | Same formula as Section 5 for fair comparison |


In [0]:
import os, time
from pathlib import Path
from landingai_ade import LandingAIADE

# -- API key from Databricks secrets (recommended) or env var
try:
    LANDING_AI_KEY = dbutils.secrets.get(scope='landing_ai', key='vision_agent_api_key')
except Exception:
    LANDING_AI_KEY = os.environ.get('vision_agent_api_key', 'YOUR_API_KEY_HERE')

client = LandingAIADE(apikey=LANDING_AI_KEY)
GROUNDING_DIR = f"{PATH_RESULTS}/landingai_groundings"
dbutils.fs.mkdirs(GROUNDING_DIR)

# -- Parse preprocessed PDF
print('Parsing with LandingAI ADE (dpt-2-latest) ...')
t0 = time.time()
ade_response = client.parse(
    document  = Path(PREPROCESSED_PDF),
    model     = 'dpt-2-latest',
    save_to   = GROUNDING_DIR,
)
elapsed_ade  = round(time.time() - t0, 2)
chunks       = ade_response.chunks
markdown_ade = ade_response.markdown or ''
print(f'  Chunks   : {len(chunks)}')
print(f'  MD chars : {len(markdown_ade):,}')
print(f'  Elapsed  : {elapsed_ade}s')



In [0]:
# -- Normalise chunks to scoring dimensions
type_counts_ade = {}
text_filled = 0
for chunk in chunks:
    ct = str(chunk.type).split('.')[-1].lower()
    type_counts_ade[ct] = type_counts_ade.get(ct, 0) + 1
    if (chunk.markdown or '').strip(): text_filled += 1

n_ade        = len(chunks)
fill_ade     = round(text_filled / n_ade * 100, 1) if n_ade > 0 else 0.0
conf_list    = [c.confidence for c in chunks if hasattr(c,'confidence') and c.confidence is not None]
avg_conf_ade = round(sum(conf_list)/len(conf_list)*100,1) if conf_list else 75.0
composite_ade = round((0.33*100 +  0.33*fill_ade + 0.33*avg_conf_ade), 1)
score_ade = {'label':'LandingAI ADE (dpt-2)','elements':n_ade,
             'no_error':100.0,'text_fill_rate':fill_ade,
             'avg_confidence':avg_conf_ade,'composite_score':composite_ade,
             'elapsed_s':elapsed_ade,'type_counts':type_counts_ade}


# -- Head-to-Head comparison
print()
print('=' * 72)
print('  HEAD-TO-HEAD  --  Preprocessed old_articles.pdf')
print('=' * 72)
hdr = '{:<28} {:>12} {:>14} {:>12}'
print(hdr.format('Metric', 'Raw baseline', 'Databricks v2.0', 'LandingAI ADE'))
print('-' * 72)
comparison = [
    ('Elements / chunks',  score_raw['elements'],         score_pre['elements'],         score_ade['elements'],         ''),
    ('No-error',           score_raw['no_error'],         score_pre['no_error'],         score_ade['no_error'],         '%'),
    ('Text fill rate',     score_raw['text_fill_rate'],   score_pre['text_fill_rate'],   score_ade['text_fill_rate'],   '%'),
    ('Avg confidence',     score_raw['avg_confidence'],   score_pre['avg_confidence'],   score_ade['avg_confidence'],   '%'),
    ('Parse time',         score_raw['elapsed_s'],        score_pre['elapsed_s'],        score_ade['elapsed_s'],        's'),
]
for name, raw, dbx, ade, unit in comparison:
    if name == 'Parse time':
        winner = 'DBX <<' if dbx <= ade else 'ADE <<'
    else:
        winner = 'DBX <<' if dbx >= ade else 'ADE <<'
    print(hdr.format(name, f'{raw}{unit}', f'{dbx}{unit}', f'{ade}{unit}') + f'  {winner}')
print('=' * 72)

print('\nLandingAI Markdown preview (first 1500 chars):')
print('-' * 60)
print(markdown_ade[:1500])
print('-' * 60)

all_types = sorted(set(list(score_pre['type_counts'].keys()) + list(type_counts_ade.keys())))
print('\nElement type distribution:')
hdr2 = '{:<22} {:>14} {:>14}'
print(hdr2.format('Type', 'Databricks v2.0', 'LandingAI ADE'))
print('-' * 52)
for t in all_types:
    print(hdr2.format(t, score_pre['type_counts'].get(t,0), type_counts_ade.get(t,0)))


### 6d. Side-by-Side Markdown Comparison

Persist both markdown outputs and render a split-panel view in the notebook.

In [0]:
# Save both markdowns to the results volume
ade_md_path = f"{PATH_RESULTS}/old_articles_landingai.md"
with open(ade_md_path, 'w', encoding='utf-8') as f:
    f.write(f'# old_articles.pdf -- LandingAI ADE (dpt-2-latest)\n\n')
    f.write(markdown_ade)
print(f'LandingAI markdown saved : {ade_md_path}')

dbx_md      = elements_to_markdown(preprocessed_parse_elems, 'old_articles_preprocessed.pdf')
dbx_md_path = f"{PATH_RESULTS}/old_articles_databricks_preprocessed.md"
with open(dbx_md_path, 'w', encoding='utf-8') as f:
    f.write(dbx_md)
print(f'Databricks markdown saved: {dbx_md_path}')

# Render split-panel HTML in notebook
def esc(s): return s.replace('&','&amp;').replace('<','&lt;').replace('>','&gt;')

displayHTML(f'''
<div style="display:flex;gap:24px;font-family:Georgia,serif;font-size:13px;line-height:1.6;">
  <div style="flex:1;padding:20px;background:#f5f9ff;border:1px solid #b3d1ff;border-radius:8px;">
    <p style="font-weight:bold;color:#1a4a9e;margin-bottom:12px;">
      Databricks ai_parse_document v2.0<br>
      <small style="font-weight:normal">{score_pre["elements"]} elements &middot; {score_pre["composite_score"]}% score</small>
    </p>
    <pre style="font-size:12px;white-space:pre-wrap;max-height:600px;overflow-y:auto">{esc(dbx_md[:3000])}</pre>
  </div>
  <div style="flex:1;padding:20px;background:#f5fff8;border:1px solid #7ecfaa;border-radius:8px;">
    <p style="font-weight:bold;color:#1a6640;margin-bottom:12px;">
      LandingAI ADE -- dpt-2-latest<br>
      <small style="font-weight:normal">{score_ade["elements"]} chunks &middot; {score_ade["composite_score"]}% score</small>
    </p>
    <pre style="font-size:12px;white-space:pre-wrap;max-height:600px;overflow-y:auto">{esc(markdown_ade[:3000])}</pre>
  </div>
</div>
''')


### 5e. PNG Dashboard — Figure Description Quality

In [0]:
png_elements = parse_results["dashboard_png"]["parsed"].get("document", {}).get("elements", [])

figures = [e for e in png_elements if e.get("type") in ("figure", "image")]
text_e  = [e for e in png_elements if e.get("type") == "text"]

print("🖼️ PNG Dashboard analysis")
print(f"   Figure elements   : {len(figures)}")
print(f"   Text elements     : {len(text_e)}")

print("\n AI-generated descriptions (figures only — v2.0 limitation):")
for fig in figures[:3]:
    desc = fig.get("description", "(none)").strip()
    print(f"   ▸ {desc[:250]}")

print("\n📝 Text extracted from image:")
for t in text_e[:5]:
    print(f"   ▸ {t.get('text','').strip()[:150]}")

## 7. Global Summary Dashboard

In [0]:
print("=" * 65)
print(" ai_parse_document — ACCURACY SUMMARY")
print("=" * 65)

header = f"{'File':<30} {'Pages':>5} {'Elems':>6} {'Score':>7} {'Time':>7}"
print(header)
print("-" * 65)

for s in sorted(scorecards, key=lambda x: -x["composite_score"]):
    bar_len = int(s["composite_score"] / 5)
    bar = "█" * bar_len + "░" * (20 - bar_len)
    grade = "🟢" if s["composite_score"] >= 80 else "🟡" if s["composite_score"] >= 55 else "🔴"
    print(f"{s['filename']:<30} {s['element_count']:>6} {s['composite_score']:>6.1f}% {s['elapsed_s']:>6.1f}s  {grade}")
    print(f"  {bar}  no_err={s['no_error']}% fill={s['text_fill_rate']}% conf={s['avg_confidence']}%")
    print()

avg = round(sum(s["composite_score"] for s in scorecards) / len(scorecards), 1)
print(f"\n{'OVERALL AVERAGE SCORE':<40} {avg:>6.1f}%")
print("=" * 65)

## 8. Save Results to Delta Table

In [0]:
TABLE_NAME = f"{catalog}.{schema}.ai_parse_accuracy_results"
run_ts = datetime.now().isoformat()

df_final = df_scorecard.withColumn("run_timestamp", F.lit(run_ts))

(
    df_final.write
    .format("delta")
    .mode("append")
    .saveAsTable(TABLE_NAME)
)

print(f"✅ Results saved to: {TABLE_NAME}")
print(f"   Run timestamp: {run_ts}")
display(spark.table(TABLE_NAME).orderBy(F.col("run_timestamp").desc(), F.col("composite_score").desc()))

## 9. Interpretation & Recommendations

### Reading the scores

| Score | Grade | Meaning |
|---|---|---|
| ≥ 80% | 🟢 Good | Reliable extraction — ready for production |
| 55–79% | 🟡 Medium | Usable but review edge-cases |
| < 55% | 🔴 Poor | Needs pre-processing or alternative approach |

### Expected results per format

- **Invoice PDF** → typically 🟢 — structured layout, dense text/table, high element coverage
- **PPTX** → typically 🟢/🟡 — good slide extraction, images may lack text
- **PNG Dashboard** → typically medium — figures well described by AI (via `descriptionElementTypes`), but native text extraction from raster images can be limited
- **Scanned PDF** → typically medium/poor — depends heavily on scan resolution and handwriting legibility

> **v2.0 note:** `descriptionElementTypes='*'` and `descriptionElementTypes='figure'` are equivalent — AI descriptions are generated for **figures only**. Table and text descriptions are not yet supported. Set `descriptionElementTypes=''` to skip descriptions entirely and reduce cost.

### Tips for improving scanned document results

```python
# Pre-process low-quality scans before parsing:
# 1. Increase DPI to ≥ 300 using PIL / OpenCV
# 2. Apply binarisation / noise removal
# 3. Correct skew before calling ai_parse_document
from PIL import Image, ImageFilter
img = Image.open(scan_path).convert('L')  # Grayscale
img = img.filter(ImageFilter.SHARPEN)
img.save(preprocessed_path, dpi=(300, 300))
```